## 1. Description

This notebook automates running and aggregating LCModel quantification results. It can launch executions over multiple signal files while varying parameter files. It uses the VIP platform to execute jobs and Girder to manage files. Finally, it can aggregate the results into a tabular file to facilitate analysis.

- Configure and authenticate against the VIP and Girder APIs
- Run LCModel jobs on signal and parameter files with VIP
- Download and extract result `.tgz` archives
- Parse LCModel `.table` files into DataFrames
- Aggregate data and add metadata (DKNTMN, file name)
- Compare quantifications (`Rate_Cr`) using boxplots

## 2. Imports & Configuration

This cell includes:

- **Standard libraries**: file, path and archive handling
- **Data science libraries**: pandas, numpy, matplotlib
- **VIP & Girder clients** for data access
- **User parameters**: API keys, URL, Girder folder IDs, local paths

In [ ]:
# 2.1 — Standard Libraries
import os
import tarfile
import shutil
from pathlib import Path

# 2.2 — Data Science
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 2.3 — VIP & Girder
from vip_client.classes import VipGirder
from girder_client import GirderClient

# 2.4 — User parameters (to fill)
VIP_API_KEY         = os.environ.get('VIP_API_KEY')
GIRDER_API_KEY      = os.environ.get('GIRDER_API_KEY')

GIRDER_URL          = "https://srmnopt.creatis.insa-lyon.fr/api/v1"
PIPELINE_ID         = "LCModel/0.2"
LAUNCH_EXECUTION    = True

# 2.5 — Pipeline parameters
CONTROL_FOLDER_PATH = '/collection/Schizemo/DATA/DATA_REPRO_VIP/inputs/parameters-DKNTMN-from-10-to-150'
SIGNAL_FOLDER_PATH  = '/collection/Schizemo/DATA/DATA_REPRO_VIP'
OUTPUT_FOLDER_PATH  = '/collection/Schizemo/DATA/DATA_REPRO_VIP/outputs-DKNTMN-from-10-to-150/old'
BASIS_FOLDER_PATH   = '/collection/Schizemo/DATA/DATA_REPRO_VIP/inputs'

# 2.6 — Local directories to save files
EXTRACTION_FOLDER   = Path('./downloaded_outputa')
EXTRACTION_FOLDER.mkdir(exist_ok=True)

## 3. VIP and Girder: Initialization

Authenticate and prepare the clients:

- **VIP** to launch sessions and retrieve results
- **GirderClient** to browse collections and download files

In [ ]:
# 3.1 — VIP
if LAUNCH_EXECUTION:
    VipGirder.init(
        vip_key=VIP_API_KEY,
        girder_key=GIRDER_API_KEY,
        girder_api_url=GIRDER_URL,
        girder_id_prefix="magicsGirder"
    )

# 3.2 — GirderClient
gc = GirderClient(apiUrl=GIRDER_URL)
gc.authenticate(apiKey=GIRDER_API_KEY)

## 4. Running LCModel jobs (Optional)

If `LAUNCH_EXECUTION` is `True`, this cell runs LCModel jobs on the specified signal and parameter files. It uses the paths provided by the user in the first cell.

In [ ]:

if LAUNCH_EXECUTION:
    control_files_list = gc.listItem(gc.resourceLookup(CONTROL_FOLDER_PATH)['_id'])
    control_files_list = list(control_files_list)

    signal_files_list = gc.listItem(gc.resourceLookup(SIGNAL_FOLDER_PATH)['_id'])
    signal_files_list = list(signal_files_list)

    for i in range(0, len(control_files_list)):
        INPUTS_SETTINGS = {
            "signal_file": [SIGNAL_FOLDER_PATH + "/" + signal_files_list[k]["name"] for k in range(len(signal_files_list))],
            "zipped_folder": BASIS_FOLDER_PATH + "/basis.zip",
            "makebasis_file": BASIS_FOLDER_PATH + "/makeBasis_3T_Mac_VIP.in",
            "control_file": CONTROL_FOLDER_PATH + "/" + control_files_list[i]["name"],    
        }
        # split this to keep only the number at the end, prefix it with DKNTMN_
        session_name = "asupr_LCMODEL_DKNTMN_" + control_files_list[i]["name"].split("_")[-1].split(".")[0]
        # create a folder in the output path with the session name
        subfolder = gc.createFolder(gc.resourceLookup(OUTPUT_FOLDER_PATH)['_id'], session_name, public=True)
        new_session = VipGirder(
            session_name = session_name,
            pipeline_id = PIPELINE_ID,
            input_settings = INPUTS_SETTINGS,
            output_dir = OUTPUT_FOLDER_PATH + '/' + session_name,
        ).run_session(nb_runs=1)
else:
    print("LAUNCH_EXECUTION is set to False, skipping the pipeline")

## 5. Utility Functions

We define here the routines for downloading, extracting and parsing:
- **download_and_extract_tgz**: handles download and extraction without blocking on Windows
- **get_table**: reads an LCModel `.table` file, returns a DataFrame and diagnostics
- **parse_lcmodel**: reformats raw data, computes CRLB and convergence status

In [ ]:
def download_and_extract_tgz(item: any, dest_dir: Path):
    """
    Download a `.tgz` from Girder (item_id) into dest_dir,
    extract its contents, then remove the archive.
    """
    tgz_path = dest_dir / f"{item['name']}.tgz"
    gc.downloadItem(item['_id'], str(tgz_path))
    
    file = os.listdir(tgz_path)[0]
    # tgz_path = tgz_path + '/' + file
    tgz_path = os.path.join(tgz_path, file)
    # Extraction
    with tarfile.open(tgz_path, mode='r:gz') as tar:
        tar.extractall(path=str(dest_dir))

    tgz_path = Path(tgz_path)
    tgz_path.unlink()

### 5.2 Reading an LCModel `.table` file

In [ ]:

def get_table(table_path) -> tuple[pd.DataFrame, list]:
    """Function to read a .table file after LCmodel execution"""

    # Create an empty data frame 
    data = pd.DataFrame()
    
    # Read the result file
    with open(table_path, 'r') as f:
        # Look for Concentration data
        while f.readline().split(' ')[0] != "$$CONC":
            pass
        # Get headers
        line = f.readline()[:-1]
        headers = list(filter(None, line.split()))
        lh = len(headers)
        # Get the data
        rawdata=[]
        line = f.readline()[:-1]
        while line :
            # Split line by unknown number of spaces
            l = list(filter(None, line.split())) 
            if len(l) < lh:
                # Problem when a +/- sign replaces the space for macromolecules
                for sep in ["+", "-"]:
                    if sep in l[-1]:
                        l = l[:-1] + l[-1].split(sep)       
            # Update the result matrix
            rawdata.append(l) 
            # Read new line
            line = f.readline()[:-1]
        # Look for the Diagnostics table
        while f.readline().split(' ')[0] != "$$DIAG":
            pass
        # Read first line after $$DIAG
        line = f.readline()[:-1]
        diag = []
        # Record each line
        while line:
            diag.append(line)
            line = f.readline()[:-1]

    # Convert the data to a dataframe
    data = pd.DataFrame(
        {
            headers[j]: [
                rawdata[i][j] for i in range(len(rawdata))
            ] for j in range(len(headers))
        }
    )
    # Return the results
    return data, diag

### 5.3 Parsing LCModel results

In [ ]:
def parse_lcmodel(data_raw, diag, exec_path) -> pd.DataFrame:
    """Reformats LCModel output and adds the convergence flag."""
    data_exec = pd.DataFrame({
        'Metabolite': data_raw['Metabolite'],
        'Rate_Raw':  data_raw['Conc.'].astype(float),
        'Rate_Cr':   data_raw['/Cr+PCr'].astype(float),
    })
    data_exec['CRLB_Raw'] = (
        data_raw['%SD'].str.rstrip('%').astype(float) *
        data_exec['Rate_Raw'] / 100
    )
    cr_val = data_exec.loc[data_exec['Metabolite']=='Cr+PCr', 'Rate_Cr'].iat[0]
    fit_ok = cr_val != 0
    diag_ok = all(
        d.split()[1].lower() in ('info','warning') for d in diag if d.strip()
    )
    return data_exec.assign(Convergence=fit_ok and diag_ok)


## 6. Downloading & Extracting Results

Iterate over each `LCMODEL_DKNTMN_` folder on Girder, then:

1. Download the `.tgz` archives
2. Extract them locally using the `download_and_extract_tgz` function


In [ ]:
# 6.1 — List DKNTMN folders on Girder
folders = list(gc.listFolder(gc.resourceLookup(OUTPUT_FOLDER_PATH)['_id']))
print(gc.resourceLookup(OUTPUT_FOLDER_PATH))
dkntmn_folders = [
    f for f in folders if f['name'].startswith('LCMODEL_DKNTMN_')
]

# 6.2 — For each folder, download + extract
for folder in dkntmn_folders:
    sub = next(gc.listFolder(folder['_id']))
    for item in gc.listItem(sub['_id']):
        if item['name'].endswith('.tgz'):
            dest = EXTRACTION_FOLDER / folder['name'] / item['name']
            download_and_extract_tgz(item, dest)
            # Move the .table file
            table_path = dest / 'result.table'
            if table_path.exists():
                # Move the .table file
                new_table_path = str(dest).split('.')[0] + '.table'
                table_path.rename(new_table_path)
            else:
                print(f"Table file not found for {item['name']} in {dest}.")
            # Remove the extracted directory
            shutil.rmtree(dest, ignore_errors=True)


## 7. Aggregation & Cleanup of `.table` files
Aggregate LCModel quantification results into a single DataFrame, written to a CSV file. This centralizes data for later analysis, avoiding repeated downloads and parsing of individual files.

1. Collect all extracted `.table` files
2. Parse each table and add `DKNTMN` and `File` columns
3. Concatenate into a single DataFrame

In [ ]:
# 7.1 — List `.table` files
tables = []
for wf in EXTRACTION_FOLDER.iterdir():
    for file in wf.glob('**/*.table'):
        tables.append((wf.name, file))

# 7.2 — Concatenate all results
all_df = []
for dkntmn, path in tables:
    raw, diag = get_table(path)
    df = parse_lcmodel(raw, diag, EXTRACTION_FOLDER)
    df['DKNTMN'] = dkntmn.split('_')[-1]
    # sort the dataframe by DKNTMN
    df = df.sort_values(by=['DKNTMN'])
    df['File']   = path.stem
    df['Group']  = path.stem.split('_')[1]
    all_df.append(df)

data_set = pd.concat(all_df, ignore_index=True)
data_set.to_csv('data_set.csv', index=False)


## 8. Save the aggregated CSV to Girder
- **Export** the aggregated DataFrame to a CSV file
- **Upload** the CSV file to Girder into the specified folder


In [ ]:
data_set.to_csv('data_set.csv', index=False)
item = gc.createItem(
    parentFolderId=gc.resourceLookup(OUTPUT_FOLDER_PATH)['_id'],
    name='data_set.csv',
    description='Aggregated LCModel results from multiple DKNTMN runs.'
)
gc.uploadFileToItem(
    itemId=item['_id'],
    filename='data_set.csv',
    filepath='./data_set.csv'
)